# Step 4b — Mutual Nearest Neighbour (MNN) Filtering
DINOv2 (finetuned) · MNN verification · comparison with soft-argmax baseline · SPair-71k

In [ ]:
# Cell 0 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_ROOT = '/content/drive/MyDrive/semantic_correspondence'
RESULTS_DIR = f'{DRIVE_ROOT}/results/step4/mnn'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Drive mounted.')

In [ ]:
# Cell 1 — Install + clone
!pip install -q timm pandas
import os, subprocess, sys
REPO_PATH = '/content/semantic_correspondence'
if not os.path.exists(REPO_PATH):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/YOUR_USER/semantic_correspondence.git',
                    REPO_PATH], check=False)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)
print('Ready.')

In [ ]:
# Cell 2 — Imports + config
import torch, numpy as np, json, time, os
import torch.nn.functional as F
import pandas as pd
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

SPAIR_BASE    = f'{DRIVE_ROOT}/datasets/SPair-71k/SPair-71k'
PAIR_ANN_PATH = f'{SPAIR_BASE}/PairAnnotation'
LAYOUT_PATH   = f'{SPAIR_BASE}/Layout'
IMAGE_PATH    = f'{SPAIR_BASE}/JPEGImages'
DATASET_SIZE  = 'large'
PCK_ALPHA     = 0.1
THRESHOLDS    = [0.05, 0.1, 0.2]
DINOV2_W      = f'{DRIVE_ROOT}/weights/dinov2_vitb14_pretrain.pth'
FINETUNED_W   = f'{DRIVE_ROOT}/weights/finetuned/dinov2_best.pth'
IMG_SIZE      = 518
PATCH_SIZE    = 14

In [ ]:
# Cell 3 — Inline utility functions
class Normalize:
    def __init__(self, image_keys):
        self.image_keys = image_keys
        self.normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    def __call__(self, sample):
        for key in self.image_keys:
            sample[key] /= 255.0
            sample[key] = self.normalize(sample[key])
        return sample

def read_img(path):
    img = np.array(Image.open(path).convert('RGB'))
    return torch.tensor(img.transpose(2, 0, 1).astype(np.float32))

class SPairDataset(Dataset):
    def __init__(self, pair_ann_path, layout_path, image_path, dataset_size, pck_alpha, datatype):
        self.ann_files = open(os.path.join(layout_path, dataset_size, datatype + '.txt')).read().split('\n')
        self.ann_files = self.ann_files[:len(self.ann_files) - 1]
        self.pair_ann_path = pair_ann_path
        self.datatype = datatype
        self.image_path = image_path
        self.transform = Normalize(['src_img', 'trg_img'])
    def __len__(self): return len(self.ann_files)
    def __getitem__(self, idx):
        ann_file = self.ann_files[idx] + '.json'
        with open(os.path.join(self.pair_ann_path, self.datatype, ann_file)) as f:
            ann = json.load(f)
        category = ann['category']
        src_img = read_img(os.path.join(self.image_path, category, ann['src_imname']))
        trg_img = read_img(os.path.join(self.image_path, category, ann['trg_imname']))
        sample = {'src_imname': ann['src_imname'], 'trg_imname': ann['trg_imname'],
                  'src_imsize': src_img.size(), 'trg_imsize': trg_img.size(),
                  'trg_bbox': ann['trg_bndbox'], 'category': category,
                  'src_img': src_img, 'trg_img': trg_img,
                  'src_kps': torch.tensor(ann['src_kps']).float(),
                  'trg_kps': torch.tensor(ann['trg_kps']).float(),
                  'kps_ids': ann['kps_ids']}
        return self.transform(sample)

def extract_dense_features(model, img_tensor):
    with torch.no_grad():
        fd = model.forward_features(img_tensor)
        pt = fd['x_norm_patchtokens']
        B, N, D = pt.shape
        H = W = int(N ** 0.5)
        return pt.reshape(B, H, W, D)

def pixel_to_patch_coord(x, y, original_size, patch_size=14, resized_size=518):
    sx = resized_size / original_size[0]
    sy = resized_size / original_size[1]
    px = int(x * sx // patch_size)
    py = int(y * sy // patch_size)
    mx = resized_size // patch_size - 1
    return min(max(px, 0), mx), min(max(py, 0), mx)

def patch_to_pixel_coord(patch_x, patch_y, original_size, patch_size=14, resized_size=518):
    xr = patch_x * patch_size + patch_size / 2
    yr = patch_y * patch_size + patch_size / 2
    return xr * original_size[0] / resized_size, yr * original_size[1] / resized_size

def find_best_match_argmax(s, width):
    idx = s.argmax().item()
    return idx % width, idx // width

def find_best_match_windowed_softargmax(similarities, width, height, K=5, temperature=1.0):
    idx = similarities.argmax().item()
    cx, cy = idx % width, idx // width
    x0 = max(cx - K // 2, 0); x1 = min(cx + K // 2 + 1, width)
    y0 = max(cy - K // 2, 0); y1 = min(cy + K // 2 + 1, height)
    window_sims = similarities.reshape(height, width)[y0:y1, x0:x1]
    weights = torch.softmax(window_sims.flatten() * temperature, dim=0)
    ys, xs = torch.meshgrid(
        torch.arange(y0, y1, dtype=torch.float32, device=similarities.device),
        torch.arange(x0, x1, dtype=torch.float32, device=similarities.device),
        indexing='ij'
    )
    return (weights * xs.flatten()).sum().item(), (weights * ys.flatten()).sum().item()

def find_best_match_mnn(similarities, src_features, tgt_features, width, height, K=5, temperature=1.0):
    """Forward soft-argmax then backward argmax for MNN verification."""
    fwd_x, fwd_y = find_best_match_windowed_softargmax(similarities, width, height, K, temperature)
    tpx = int(round(fwd_x))
    tpy = int(round(fwd_y))
    tpx = min(max(tpx, 0), width - 1)
    tpy = min(max(tpy, 0), height - 1)
    D = tgt_features.shape[-1]
    tgt_patch = tgt_features[0, tpy, tpx, :]
    src_flat = src_features.reshape(height * width, D)
    src_norm = F.normalize(src_flat, dim=1)
    tgt_norm = F.normalize(tgt_patch.unsqueeze(0), dim=1)
    back_sim = (tgt_norm @ src_norm.T).squeeze(0)
    back_idx = back_sim.argmax().item()
    back_x = back_idx % width
    back_y = back_idx // width
    return fwd_x, fwd_y, back_x, back_y

def apply_mnn_filter(fwd_x, fwd_y, back_x, back_y,
                      src_patch_x, src_patch_y,
                      similarities, src_features, tgt_features,
                      width, height, K=5, temperature=1.0, max_patch_dist=1):
    """Return verified match or soft-argmax fallback if MNN check fails."""
    dist = max(abs(back_x - src_patch_x), abs(back_y - src_patch_y))
    if dist <= max_patch_dist:
        return fwd_x, fwd_y
    # MNN failed: mask the forward position and rerun soft-argmax
    sims_masked = similarities.clone()
    tpx = int(round(fwd_x))
    tpy = int(round(fwd_y))
    tpx = min(max(tpx, 0), width - 1)
    tpy = min(max(tpy, 0), height - 1)
    sims_masked[tpy * width + tpx] = -1e9
    fb_x, fb_y = find_best_match_windowed_softargmax(sims_masked, width, height, K, temperature)
    return fb_x, fb_y

def compute_pck_spair71k(pred_points, gt_points, bbox, threshold):
    pred = np.array(pred_points)
    gt   = np.array(gt_points)
    dist = np.sqrt(np.sum((pred - gt) ** 2, axis=1))
    norm = max(bbox[2] - bbox[0], bbox[3] - bbox[1])
    nd   = dist / norm
    mask = nd <= threshold
    return float(np.mean(mask) * 100), mask, nd

def evaluate_with_mnn(model, dataset, device, thresholds, patch_size, resized_size,
                       K=5, temperature=1.0, max_patch_dist=1):
    per_img = []
    model.eval()
    with torch.no_grad():
        for idx, sample in enumerate(dataset):
            src_t = F.interpolate(sample['src_img'].unsqueeze(0).to(device),
                                   size=(resized_size, resized_size), mode='bilinear', align_corners=False)
            tgt_t = F.interpolate(sample['trg_img'].unsqueeze(0).to(device),
                                   size=(resized_size, resized_size), mode='bilinear', align_corners=False)
            src_sz = (sample['src_imsize'][2], sample['src_imsize'][1])
            tgt_sz = (sample['trg_imsize'][2], sample['trg_imsize'][1])
            sf = extract_dense_features(model, src_t)
            tf = extract_dense_features(model, tgt_t)
            _, H, W, D = tf.shape
            tf_flat = tf.reshape(H * W, D)
            src_kps = sample['src_kps'].numpy()
            trg_kps = sample['trg_kps'].numpy()
            trg_bbox = sample['trg_bbox']
            preds = []
            for i in range(src_kps.shape[0]):
                px, py = pixel_to_patch_coord(src_kps[i,0], src_kps[i,1], src_sz, patch_size, resized_size)
                sim = F.cosine_similarity(sf[0, py, px, :].unsqueeze(0), tf_flat, dim=1)
                fwd_x, fwd_y, back_x, back_y = find_best_match_mnn(
                    sim, sf, tf, W, H, K=K, temperature=temperature
                )
                mx, my = apply_mnn_filter(
                    fwd_x, fwd_y, back_x, back_y, px, py,
                    sim, sf, tf, W, H, K=K, temperature=temperature, max_patch_dist=max_patch_dist
                )
                rx, ry = patch_to_pixel_coord(mx, my, tgt_sz, patch_size, resized_size)
                preds.append([rx, ry])
            pcks = {}
            for thr in thresholds:
                pck, _, _ = compute_pck_spair71k(preds, trg_kps.tolist(), trg_bbox, thr)
                pcks[thr] = pck
            per_img.append({'category': sample['category'], 'pck_scores': pcks})
            if (idx + 1) % 200 == 0:
                print(f'  {idx+1} pairs...')
    return per_img

def evaluate_with_softargmax(model, dataset, device, thresholds, patch_size,
                               resized_size, K=5, temperature=1.0):
    per_img = []
    model.eval()
    with torch.no_grad():
        for idx, sample in enumerate(dataset):
            src_t = F.interpolate(sample['src_img'].unsqueeze(0).to(device),
                                   size=(resized_size, resized_size), mode='bilinear', align_corners=False)
            tgt_t = F.interpolate(sample['trg_img'].unsqueeze(0).to(device),
                                   size=(resized_size, resized_size), mode='bilinear', align_corners=False)
            src_sz = (sample['src_imsize'][2], sample['src_imsize'][1])
            tgt_sz = (sample['trg_imsize'][2], sample['trg_imsize'][1])
            sf = extract_dense_features(model, src_t)
            tf = extract_dense_features(model, tgt_t)
            _, H, W, D = tf.shape
            tf_flat = tf.reshape(H * W, D)
            src_kps = sample['src_kps'].numpy()
            trg_kps = sample['trg_kps'].numpy()
            trg_bbox = sample['trg_bbox']
            preds = []
            for i in range(src_kps.shape[0]):
                px, py = pixel_to_patch_coord(src_kps[i,0], src_kps[i,1], src_sz, patch_size, resized_size)
                sim = F.cosine_similarity(sf[0, py, px, :].unsqueeze(0), tf_flat, dim=1)
                mx, my = find_best_match_windowed_softargmax(sim, W, H, K=K, temperature=temperature)
                rx, ry = patch_to_pixel_coord(mx, my, tgt_sz, patch_size, resized_size)
                preds.append([rx, ry])
            pcks = {}
            for thr in thresholds:
                pck, _, _ = compute_pck_spair71k(preds, trg_kps.tolist(), trg_bbox, thr)
                pcks[thr] = pck
            per_img.append({'category': sample['category'], 'pck_scores': pcks})
            if (idx + 1) % 200 == 0:
                print(f'  {idx+1} pairs...')
    return per_img

def save_exp_results(per_img, name, thresholds, results_dir):
    stats = {}
    for thr in thresholds:
        pcks = [m['pck_scores'][thr] for m in per_img]
        stats[f'pck@{thr:.2f}'] = {'mean': float(np.mean(pcks)), 'std': float(np.std(pcks))}
        print(f'  PCK@{thr:.2f}: {np.mean(pcks):.2f}% ± {np.std(pcks):.2f}%')
    out = {'name': name, 'n_pairs': len(per_img), 'stats': stats}
    path = os.path.join(results_dir, f'{name}.json')
    with open(path, 'w') as f: json.dump(out, f, indent=2)
    print(f'  Saved -> {path}')
    return stats

print('Utility functions loaded.')

In [ ]:
# Cell 4 — Load DINOv2 (finetuned if available)
from src.models.dinov2.dinov2.models.vision_transformer import vit_base as dinov2_vit_base

dinov2 = dinov2_vit_base(
    img_size=(518, 518), patch_size=14,
    num_register_tokens=0, block_chunks=0, init_values=1.0
).to(device)

if os.path.exists(FINETUNED_W):
    ckpt = torch.load(FINETUNED_W, map_location=device)
    dinov2.load_state_dict(ckpt['model_state_dict'], strict=False)
    print(f'Loaded finetuned weights.')
else:
    ckpt = torch.load(DINOV2_W, map_location=device)
    dinov2.load_state_dict(ckpt, strict=True)
    print(f'Loaded pretrained weights.')

dinov2.eval()
test_ds = SPairDataset(PAIR_ANN_PATH, LAYOUT_PATH, IMAGE_PATH, DATASET_SIZE, PCK_ALPHA, datatype='test')
print(f'Test set: {len(test_ds)} pairs')

In [ ]:
# Cell 5 — Soft-argmax baseline (best K and T from step 3, or default)
BEST_K = 5
BEST_T = 1.0
print(f'=== Soft-argmax baseline (K={BEST_K}, T={BEST_T}) ===')
res_base = evaluate_with_softargmax(
    dinov2, test_ds, device, THRESHOLDS, PATCH_SIZE, IMG_SIZE, K=BEST_K, temperature=BEST_T
)
stats_base = save_exp_results(res_base, 'mnn_softargmax_baseline', THRESHOLDS, RESULTS_DIR)

In [ ]:
# Cell 6 — MNN ablation: max_patch_dist ∈ {0, 1, 2, 3}
dist_list = [0, 1, 2, 3]
mnn_dist_results = []

for mpd in dist_list:
    print(f'\n--- MNN max_patch_dist={mpd} ---')
    res = evaluate_with_mnn(
        dinov2, test_ds, device, [0.1], PATCH_SIZE, IMG_SIZE,
        K=BEST_K, temperature=BEST_T, max_patch_dist=mpd
    )
    mean_pck = float(np.mean([m['pck_scores'][0.1] for m in res]))
    mnn_dist_results.append({'max_patch_dist': mpd, 'pck@0.10': mean_pck})
    print(f'  PCK@0.10={mean_pck:.2f}%')

best_mpd = max(mnn_dist_results, key=lambda x: x['pck@0.10'])['max_patch_dist']
print(f'\nBest max_patch_dist={best_mpd}')

In [ ]:
# Cell 7 — Full MNN evaluation with best max_patch_dist
print(f'=== Full MNN eval (K={BEST_K}, T={BEST_T}, max_patch_dist={best_mpd}) ===')
res_mnn = evaluate_with_mnn(
    dinov2, test_ds, device, THRESHOLDS, PATCH_SIZE, IMG_SIZE,
    K=BEST_K, temperature=BEST_T, max_patch_dist=best_mpd
)
stats_mnn = save_exp_results(res_mnn, f'mnn_K{BEST_K}_T{BEST_T}_d{best_mpd}', THRESHOLDS, RESULTS_DIR)

In [ ]:
# Cell 8 — Summary table
# MNN ablation by distance
df_dist = pd.DataFrame(mnn_dist_results)
print('MNN ablation by max_patch_dist:')
print(df_dist.to_string(index=False))

# Comparison: soft-argmax vs MNN
rows = [
    {
        'Method': f'Soft-argmax (K={BEST_K}, T={BEST_T})',
        'PCK@0.05': f"{stats_base.get('pck@0.05',{}).get('mean',0):.2f}%",
        'PCK@0.10': f"{stats_base.get('pck@0.10',{}).get('mean',0):.2f}%",
        'PCK@0.20': f"{stats_base.get('pck@0.20',{}).get('mean',0):.2f}%",
    },
    {
        'Method': f'MNN (K={BEST_K}, T={BEST_T}, d={best_mpd})',
        'PCK@0.05': f"{stats_mnn.get('pck@0.05',{}).get('mean',0):.2f}%",
        'PCK@0.10': f"{stats_mnn.get('pck@0.10',{}).get('mean',0):.2f}%",
        'PCK@0.20': f"{stats_mnn.get('pck@0.20',{}).get('mean',0):.2f}%",
    },
]
df_summary = pd.DataFrame(rows)
print('\n=== Step 4b Summary ===')
print(df_summary.to_string(index=False))
df_summary.to_csv(f'{RESULTS_DIR}/step4b_summary.csv', index=False)
print(f'Saved to {RESULTS_DIR}/step4b_summary.csv')